# **Preparation Notebook**



---
## Setup Environment

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd

from utstd.folders import *
from utstd.ipyrenders import *

at = AtFolder(
    course_code=36106,
    assignment="AT3",
)
at.run()

import warnings
warnings.simplefilter(action='ignore')

---
## Student Information

In [ ]:
group_name = "Neural Minds"
student_name = "Tahreema Rahman Zariyat"
student_id = "26016360"

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='group_name', value=group_name)

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_name', value=student_name)

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

### 0.b Import Packages

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
import pandas as pd
import altair as alt

---
## A. Feature Selection


## A.0 Load Data

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Load datasets
try:
  customer_df = pd.read_csv(at.folder_path / "customer.csv")
  person_df = pd.read_csv(at.folder_path / "person.csv")
  product_category_df = pd.read_csv(at.folder_path / "product_category.csv")
  product_cost_history_df = pd.read_csv(at.folder_path / "product_cost_history.csv")
  product_list_price_history_df = pd.read_csv(at.folder_path / "product_list_price_history.csv")
  product_sub_category_df = pd.read_csv(at.folder_path / "product_sub_category.csv")
  product_df = pd.read_csv(at.folder_path / "product.csv")
  sales_order_detail_df = pd.read_csv(at.folder_path / "sales_order_detail.csv")
  sales_order_header_df = pd.read_csv(at.folder_path / "sales_order_header.csv")
  sales_territory_df = pd.read_csv(at.folder_path / "sales_territory.csv")
  special_offer_product_df = pd.read_csv(at.folder_path / "special_offer_product.csv")
  special_offer_df = pd.read_csv(at.folder_path / "special_offer.csv")
  store_df = pd.read_csv(at.folder_path / "store.csv")
  unit_measure_df = pd.read_csv(at.folder_path / "unit_measure.csv")
except Exception as e:
  print(e)

### A.1 Approach 1

In [ ]:
df = sales_order_detail_df.copy()
print(df.columns.tolist())

print("\nShape:", df.shape)

print("\nDuplicate rows:", df.duplicated().sum())

print()
df.head()

In [ ]:
num_cols = ["order_quantity","unit_price","unit_price_discount","line_total"]
corr = df[num_cols].corr()
print(corr["line_total"].sort_values(ascending=False))

In [ ]:
feature_selection_1_insights = """Approach 1 looks at the correlation between numeric columns and line_total. It can be understood that unit_price and line_total are very highly correlated, which makes sense because line_total = order_quantity * unit_price * (1 - unit_price_discount). This is an exact arithmetic relationship (the maximum difference is ~1e-12 across the whole dataset). So if I keep unit_price and unit_price_discount as inputs, the "prediction" is really just a calculator and the problem is trivial. Therefore, I will drop unit_price and unit_price_discount from the feature set and let the model learn to predict line_total from product_id, special_offer_id and order_quantity along with the engineered features)."""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_1_insights', value=feature_selection_1_insights)

### A.2 Approach 2

In [ ]:
prod_stats = df.groupby("product_id")["line_total"].agg(["mean","std","count"]).reset_index()
print("Number of products:", len(prod_stats))

In [ ]:
print("Top 5 products by avg line_total:")
print(prod_stats.sort_values("mean", ascending=False).head())

In [ ]:
offer_stats = df.groupby("special_offer_id")["line_total"].mean().sort_values(ascending=False)
print(offer_stats)

In [ ]:
feature_selection_2_insights = """
Approach 2 checks whether the non-deterministic columns (product_id, special_offer_id, order_quantity) actually carry any signal. Here, product_id has huge variation in average line_total across the 254 products from a few dollars up to approximately $3,500. This is clearly informative but the high cardinality means I will need a smart encoding, not just one-hot. Next, special_offer_id has 10 levels with different average line_totals which is useful and small enough for one-hot encoding. FInally, the order_quantity directly multiplies into line_total which is obviously useful. These three columns form my candidate feature set.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_2_insights', value=feature_selection_2_insights)

### A.n Approach 3

In [ ]:
print("Unique sales_order_detail_id == rows?", df["sales_order_detail_id"].nunique() == len(df))

print("Unique sales_order_id:", df["sales_order_id"].nunique(), "out of", len(df), "rows")

In [ ]:
feature_selection_n_insights = """
Approach 3 accomplishes sanity check on IDs. sales_order_detail_id is unique per row and it is just a row identifier. sales_order_id has 15,357 unique values across 42,100 rows, so each order has approximately 2.7 lines on average. Neither column is predictive of line_total in a generalizable way. As a result, I will drop both from the feature set.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_n_insights', value=feature_selection_n_insights)

### A.z Final Selection of Features

In [ ]:
features_list = ["product_id", "special_offer_id", "order_quantity"]
target_name = "line_total"

print("Duplicate rows in full data:", df.duplicated().sum())

print("\nShape before selection:", df.shape)

keep_cols = features_list + [target_name]
df = df[keep_cols].copy()

print("\nKept columns:", df.columns.tolist())

print("\nShape after selection:", df.shape)

In [ ]:
feature_selection_explanations = """
Final feature set has 3 features. First one is product_id that is which product was sold and it has 254 unique values. Next is special_offer_id that shows which promotion was applied and this one has 10 unique values. The last one is order_quantity and it shows how many units were sold on the line. The target is line_total which will show dollar amount of the line. The target is kept here too so it gets carried through cleaning and splitting. We will separate it from X later. Moreover, I have deliberately excluded unit_price and unit_price_discount because they recreate the target arithmetically. The remaining features carry real predictive signal that the model has to learn, that is each product has its own typical price band.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_explanations', value=feature_selection_explanations)

---
## B. Data Cleaning

### B.1 Fixing duplicates

In [ ]:
print("After feature selection, repeated feature-combo rows:", df.duplicated().sum())

print("\nShape unchanged:", df.shape)

In [ ]:
data_cleaning_1_explanations = """
The full original dataset was checked for duplicate rows using all 8 columns including the unique sales_order_detail_id and found zero duplicates. After selecting only the 4 columns, many rows look the same (e.g. lots of single-unit orders of the same product under the same offer). These are not data errors. They are indeed different real transactions.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_1_explanations', value=data_cleaning_1_explanations)

### B.2 Fixing invalid target values

In [ ]:
print("line_total <= 0:", (df["line_total"] <= 0).sum())

print("\norder_quantity <= 0:", (df["order_quantity"] <= 0).sum())

df = df[(df["line_total"] > 0) & (df["order_quantity"] > 0)].reset_index(drop=True)
print("\nShape after filter:", df.shape)

In [ ]:
data_cleaning_2_explanations = """
A regression target of $0 or negative would be invalid for our use case. Same for an order_quantity <= 0. So, they were filtered out. Even though none were found in this dataset, the check protects against bad rows in future runs.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_2_explanations', value=data_cleaning_2_explanations)

### B.3 Reviewing extreme line_total values

In [ ]:
top = df["line_total"].quantile([0.95, 0.99, 0.999, 1.0])
print(top)

print("\nLines above the 99th percentile make up", round((df["line_total"] > top.loc[0.99]).mean()*100, 2), "% of rows")

In [ ]:
data_cleaning_3_explanations = """
The top 1% of line_totals are approximately above $5,000 with a maximum of $23,668 approximately. These are legitimate large orders, not data errors, so they were kept. They do mean the target distribution has a long right tail. This will be dealt with that by also reporting MAE alongside RMSE in the baseline / model notebooks and by considering a log transform on the target.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_3_explanations', value=data_cleaning_3_explanations)

### B.n Fixing "\<describe_issue_here\>"

> You can add more cells related to other issues in this section

In [ ]:
print("Cleaning is complete. \nFinal shape:", df.shape)

In [ ]:
data_cleaning_n_explanations = """
No additional cleaning steps were needed beyond the duplicate / invalid target handling. The dataset has no missing values in the columns that we are using.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_n_explanations', value=data_cleaning_n_explanations)

---
## C. Split Datasets


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

In [ ]:
train_part, temp_part = train_test_split(df, test_size=0.30, random_state=42)
validation_df, testing_df = train_test_split(temp_part, test_size=0.50, random_state=42)

training_df = train_part.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
testing_df = testing_df.reset_index(drop=True)

print("Train:", training_df.shape)
print("Val  :", validation_df.shape)
print("Test :", testing_df.shape)

In [ ]:
data_splitting_explanations = """
Here, a random split is chosen over the temporal one because the dataset does not have a time component baked into the rows we are using. I used a random 70/15/15 split with random_state=42 for reproducibility. The training set (~70%) is to fit models and compute statistics for target-mean encoding of product_id . It must be train-only to avoid leakage. The validation set (~15%) is to tune hyperparameters and compare algorithms. The test set (~15%) is touched only once to report the final performance.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_splitting_explanations', value=data_splitting_explanations)

---
## D. Feature Engineering

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Create copy of datasets

try:
  training_df_eng = training_df.copy()
  validation_df_eng = validation_df.copy()
  testing_df_eng = testing_df.copy()
except Exception as e:
  print(e)

### D.1 New Feature `product_avg_price`

In [ ]:
prod_avg = training_df_eng.groupby("product_id")["line_total"].mean()
global_avg = training_df_eng["line_total"].mean()

print("Computed avg line_total for", len(prod_avg), "products.")
print("Global avg (used for unseen products):", round(global_avg, 2))

In [ ]:
training_df_eng["product_avg_price"] = training_df_eng["product_id"].map(prod_avg).fillna(global_avg)
validation_df_eng["product_avg_price"] = validation_df_eng["product_id"].map(prod_avg).fillna(global_avg)
testing_df_eng["product_avg_price"] = testing_df_eng["product_id"].map(prod_avg).fillna(global_avg)

print(training_df_eng[["product_id","product_avg_price"]].head())

In [ ]:
feature_engineering_1_explanations = """
product_id has 254 levels which is too many for clean one-hot encoding. Additionally, the model would struggle with that many sparse columns. Instead, I encode each product_id by the mean line_total of that product in the training set (target-mean encoding). This gives the model a strong numeric prior about how expensive that product typically is. The mean is computed from the training set only, then mapped onto validation and test. Any product not seen in training falls back to the global training mean and this prevents data leakage.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_1_explanations', value=feature_engineering_1_explanations)

### D.2 New Feature `is_bulk`

In [ ]:
for d in [training_df_eng, validation_df_eng, testing_df_eng]:
    d["is_bulk"] = (d["order_quantity"] > 1).astype(int)

print(training_df_eng["is_bulk"].value_counts())

In [ ]:
feature_engineering_2_explanations = """
is_bulk is a binary flag which is 1 if order_quantity > 1, else 0. It captures the main split in the data (single-unit purchase vs. multi-unit) in a single easy feature. It is highly correlated with order_quantity but gives linear models a simpler signal to work with.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_2_explanations', value=feature_engineering_2_explanations)

### D.3 New Feature `log_order_quantity`

In [ ]:
for d in [training_df_eng, validation_df_eng, testing_df_eng]:
    d["log_order_quantity"] = np.log1p(d["order_quantity"])

print(training_df_eng[["order_quantity","log_order_quantity"]].describe())

In [ ]:
feature_engineering_3_explanations = """
order_quantity is skewed (range 1-36, mostly 1). log(1+order_quantity) gives a smoother, less skewed version, which generally helps linear models like Ridge. Tree-based models will not care either way.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_3_explanations', value=feature_engineering_3_explanations)

### D.n Fixing "\<describe_issue_here\>"

> You can add more cells related to new features in this section

In [ ]:
print("Feature engineering is complete.")
print("Now the columns are:", training_df_eng.columns.tolist())

In [ ]:
feature_engineering_n_explanations = """
Three new engineered features (i.e. product_avg_price, is_bulk, and log_order_quantity) were enough to give the model a good signal.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_n_explanations', value=feature_engineering_n_explanations)

---
## E. Data Preparation for Modeling

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Create copy of datasets

try:
  X_train = training_df_eng.copy()
  X_val = validation_df_eng.copy()
  X_test = testing_df_eng.copy()
except Exception as e:
  print(e)

### E.1 Data Transformation

In [ ]:
offer_dummies_train = pd.get_dummies(X_train["special_offer_id"], prefix="offer")
offer_cols = offer_dummies_train.columns.tolist()

def add_offer_dummies(d):
    dummies = pd.get_dummies(d["special_offer_id"], prefix="offer")
    for col in offer_cols:
        d[col] = dummies[col] if col in dummies.columns else 0
    return d

X_train = add_offer_dummies(X_train)
X_val   = add_offer_dummies(X_val)
X_test  = add_offer_dummies(X_test)

print("Created", len(offer_cols), "one-hot columns for special_offer_id.")

In [ ]:
data_transformation_1_explanations = """
special_offer_id has only 10 distinct values. So, one-hot encoding is fine, there will be no high-cardinality problem. The column list from the training set was deliberately fixed, so that validation/test get the same columns even if some offers are missing in those splits, which would otherwise break the model.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_1_explanations', value=data_transformation_1_explanations)

### E.2 Data Transformation

In [ ]:
y_train = X_train["line_total"].copy()
y_val = X_val["line_total"].copy()
y_test = X_test["line_total"].copy()

drop_cols = ["product_id", "special_offer_id", "line_total"]
X_train = X_train.drop(columns=drop_cols)
X_val = X_val.drop(columns=drop_cols)
X_test = X_test.drop(columns=drop_cols)

print("Final feature columns:")
print(X_train.columns.tolist())

print("\nShapes -> X_train:", X_train.shape, " y_train:", y_train.shape)

In [ ]:
data_transformation_2_explanations = """
After one-hot encoding, the raw special_offer_id string column is no longer needed and product_id has been replaced by product_avg_price. The target line_total is moved out of the feature matrix and into y_train/y_val/y_test. Additionally, X_train now contains only numeric features ready for any sklearn model.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_2_explanations', value=data_transformation_2_explanations)

### E.3 Data Transformation

In [ ]:
y_train.to_csv(at.folder_path / "y_train.csv", index=False)
y_val.to_csv(at.folder_path / "y_val.csv", index=False)
y_test.to_csv(at.folder_path / "y_test.csv", index=False)
print("Saved y_train.csv, y_val.csv, y_test.csv")

In [ ]:
data_transformation_3_explanations = """
The target arrays are saved as separate CSV files so that the downstream baseline and regression notebooks can load X and y independently. However, the "DO NOT MODIFY" save block at the bottom of this notebook already saves X_train/X_val/X_test as CSVs.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_3_explanations', value=data_transformation_3_explanations)

### E.n Fixing "\<describe_issue_here\>"

> You can add more cells related to data preparation in this section

In [ ]:
print("X_train preview:")
X_train.head()

In [ ]:
data_transformation_n_explanations = """
No further transformations needed. Datasets are ready for modelling. All transformations are train/val/test consistent. Feature columns are now all numeric with no missing values and the target is stored separately. The final X matrices and y arrays are saved to disk for the baseline and regression notebooks.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_n_explanations', value=data_transformation_n_explanations)

---
## F. Save Datasets

> Do not change this code

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL

try:
  X_train.to_csv(at.folder_path / 'X_train.csv', index=False)

  X_val.to_csv(at.folder_path / 'X_val.csv', index=False)

  X_test.to_csv(at.folder_path / 'X_test.csv', index=False)
except Exception as e:
  print(e)